# Phase 4 — Full-Scale Training on EchoNet-Dynamic (free-Colab safe)

This reuses the exact pipeline validated on the subsample (notebooks 02 + 03), now on the **full 10,030 videos**, with the official train/val/test split instead of a random subsample split.

**Free-tier safety measures baked in:**
- Frame extraction and quality scoring are **resumable** — safe to stop and re-run across multiple Colab sessions.
- Training checkpoints save **every N minutes**, not just every epoch, since one epoch on ~7,465 videos may not fit in a single session.
- A disk-space check runs first; if Drive is nearly full, the notebook tells you to delete the zip before continuing.
- Everything logs progress so you always know where a run left off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, time, shutil
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
VIDEOS_DIR = os.path.join(DATA_DIR, 'Videos')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
FRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/full', 'frames')
os.makedirs(FRAME_CACHE_DIR, exist_ok=True)

assert os.path.exists(FULL_CSV), f'FileList.csv not found at {FULL_CSV}'
assert os.path.isdir(VIDEOS_DIR), f'Videos folder not found at {VIDEOS_DIR}'

df = pd.read_csv(FULL_CSV)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
split_col = 'Split' if 'Split' in df.columns else None
HFrEF_THRESHOLD = 40
print('Full dataset:', df.shape)
print(df[split_col].value_counts() if split_col else 'No Split column found -- will make our own split.')

## Disk space check

Free Google Drive is 15GB total (shared with Gmail/Photos). The zip (~7GB) plus its extracted copy can eat most of that. If space is tight, delete the zip -- the extracted `Videos/` folder is all that's needed from here on.

In [ ]:
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
free_gb = free / (1024**3)
print(f'Free space on Drive (approx, via mount): {free_gb:.1f} GB')

zip_path = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic.zip')
if os.path.exists(zip_path):
    zip_gb = os.path.getsize(zip_path) / (1024**3)
    print(f'Found the original zip, still taking up {zip_gb:.1f} GB: {zip_path}')
    if free_gb < 5:
        print('Free space is low -- recommend deleting the zip now, e.g.:')
        print(f"  os.remove('{zip_path}')")
    else:
        print('Space looks OK for now, but you can delete the zip anytime once extraction is confirmed complete.')
else:
    print('Zip not found (already deleted or moved) -- fine, we only need the extracted Videos/ folder.')

## Resumable frame extraction (middle frame per video, full dataset)

Safe to interrupt and re-run -- already-cached frames are skipped. Prints periodic progress so you can gauge how much of a session you'll need.

In [ ]:
!pip install --quiet opencv-python-headless
import cv2

def get_middle_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    mid = max(0, n_frames // 2)
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

start_time = time.time()
cached, missing, skipped = 0, 0, 0
for i, row in df.iterrows():
    fname = str(row[filename_col])
    if not fname.endswith('.avi'):
        fname = fname + '.avi'
    cache_path = os.path.join(FRAME_CACHE_DIR, fname.replace('.avi', '.npy'))
    if os.path.exists(cache_path):
        skipped += 1
        continue
    video_path = os.path.join(VIDEOS_DIR, fname)
    if not os.path.exists(video_path):
        missing += 1
        continue
    frame = get_middle_frame(video_path)
    if frame is not None:
        np.save(cache_path, frame)
        cached += 1
    else:
        missing += 1
    if (cached + skipped) % 500 == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(f'[{i+1}/{len(df)}] newly cached={cached}, already done={skipped}, missing={missing}, elapsed={elapsed_min:.1f} min')

print(f'DONE this run. newly cached={cached}, already done={skipped}, missing={missing}')
print('If this session gets disconnected partway, just re-run this cell -- it will skip everything already cached.')

## Quality score (same method as notebook 03, resumable via a CSV cache)

In [ ]:
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')

def compute_quality_score(frame_rgb):
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    contrast = gray.std()
    return sharpness, contrast

if os.path.exists(quality_csv_path):
    quality_df = pd.read_csv(quality_csv_path)
    done_files = set(quality_df[filename_col])
    print(f'Resuming quality scoring -- {len(done_files)} already done.')
else:
    quality_df = pd.DataFrame(columns=[filename_col, 'sharpness', 'contrast'])
    done_files = set()

new_rows = []
for i, row in df.iterrows():
    fname_key = row[filename_col]
    if fname_key in done_files:
        continue
    fname = str(fname_key)
    if not fname.endswith('.avi'):
        fname = fname + '.avi'
    cache_path = os.path.join(FRAME_CACHE_DIR, fname.replace('.avi', '.npy'))
    if not os.path.exists(cache_path):
        continue
    frame = np.load(cache_path)
    sharpness, contrast = compute_quality_score(frame)
    new_rows.append({filename_col: fname_key, 'sharpness': sharpness, 'contrast': contrast})
    if len(new_rows) % 1000 == 0:
        quality_df = pd.concat([quality_df, pd.DataFrame(new_rows)], ignore_index=True)
        quality_df.to_csv(quality_csv_path, index=False)
        new_rows = []
        print(f'Checkpointed quality scores at {len(quality_df)} rows')

if new_rows:
    quality_df = pd.concat([quality_df, pd.DataFrame(new_rows)], ignore_index=True)
    quality_df.to_csv(quality_csv_path, index=False)

print('Quality scoring done:', quality_df.shape)

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)

df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')
print(df['quality_score'].describe())

## Build the dataset using the OFFICIAL split (not a random one, now that we have the full data)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(FRAME_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
print(f'Usable videos: {len(clean_df)} / {len(df)}')

if split_col:
    train_df = clean_df[clean_df[split_col].str.upper() == 'TRAIN'].reset_index(drop=True)
    val_df = clean_df[clean_df[split_col].str.upper() == 'VAL'].reset_index(drop=True)
    test_df = clean_df[clean_df[split_col].str.upper() == 'TEST'].reset_index(drop=True)
else:
    from sklearn.model_selection import train_test_split
    train_df, rest_df = train_test_split(clean_df, test_size=0.25, random_state=42)
    val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=42)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

class EchoFrameDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        aug = [T.RandomHorizontalFlip()] if train else []
        self.transform = T.Compose([T.ToPILImage(), T.Resize((112, 112)), *aug, T.ToTensor(), T.Normalize(mean, std)])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        frame = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        img = self.transform(frame)
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return img, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 32  # a bit larger than the subsample run since the full dataset benefits from it; lower this if you hit an out-of-memory error
train_ds = EchoFrameDataset(train_df, FRAME_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoFrameDataset(val_df, FRAME_CACHE_DIR, filename_col, ef_col, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('DataLoaders ready.')

In [ ]:
# Reproducibility: fix random seeds (used for all runs from this version onward)
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Note: GPU kernels (cuDNN) can still be non-deterministic, so re-runs may differ slightly in the last digits.

## Model (same MC-Dropout architecture as notebook 03)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable Runtime > Change runtime type > GPU (T4) and re-run from the top -- full-scale on CPU is not practical.')

class EchoMCDropout(nn.Module):
    def __init__(self, dropout_p=0.3):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        n_feat = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_head = nn.Linear(n_feat, 1)
        self.cls_head = nn.Linear(n_feat, 1)

    def forward(self, x):
        feat = self.dropout(self.backbone(x))
        return self.ef_head(feat).squeeze(-1), self.cls_head(feat).squeeze(-1)

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

EXP_NAME = 'mc_dropout_full_v1'
model = EchoMCDropout(dropout_p=0.3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.L1Loss()
print(f'HFrEF positive rate (train): {pos_rate:.3f} -> pos_weight: {pos_weight.item():.2f}')

## Time-based checkpointing (critical for free-Colab on the full dataset)

Saves every `CHECKPOINT_EVERY_MINUTES` **within** an epoch, not just at epoch end -- a full epoch over ~7,465 videos may take longer than a single Colab session allows before disconnecting.

In [ ]:
CHECKPOINT_EVERY_MINUTES = 15
STATE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_NAME, 'latest.pt')
os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)

def save_state(model, optimizer, epoch, step_in_epoch, note=''):
    torch.save({
        'epoch': epoch, 'step_in_epoch': step_in_epoch,
        'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
    }, STATE_PATH)
    log_path = os.path.join(PROJECT_ROOT, 'logs', f'{EXP_NAME}_checkpoints.jsonl')
    with open(log_path, 'a') as f:
        f.write(json.dumps({'time': time.time(), 'epoch': epoch, 'step_in_epoch': step_in_epoch, 'note': note}) + '\n')
    print(f'[checkpoint] {note} epoch={epoch} step={step_in_epoch}')

def load_state(model, optimizer):
    if not os.path.exists(STATE_PATH):
        print('[checkpoint] none found, starting from scratch')
        return 0, 0
    state = torch.load(STATE_PATH, map_location=device)
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    print(f"[checkpoint] resumed from epoch={state['epoch']} step={state['step_in_epoch']}")
    return state['epoch'], state['step_in_epoch']

start_epoch, resume_step = load_state(model, optimizer)

## Train

Modest epoch count to start (`N_EPOCHS = 5`) -- with the full dataset this is a much stronger signal than 15 epochs on 144 subsample images. Increase later if val metrics are still clearly improving and you have session time to spare.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error

N_EPOCHS = 5
last_checkpoint_time = time.time()

def evaluate(loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for imgs, ef_true, cls_true, _q in loader:
            imgs = imgs.to(device)
            ef_pred, cls_logit = model(imgs)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_pred.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    f1 = f1_score(cls_true_all, [1 if p > 0.5 else 0 for p in cls_prob_all], zero_division=0)
    return {'mae': mae, 'auc': auc, 'f1': f1}

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    running_loss, n_seen = 0.0, 0
    for step, (imgs, ef_true, cls_true, _q) in enumerate(train_loader):
        if epoch == start_epoch and step < resume_step:
            continue  # skip steps already done in this epoch before the last disconnect
        imgs, ef_true, cls_true = imgs.to(device), ef_true.to(device), cls_true.to(device)
        ef_pred, cls_logit = model(imgs)
        loss = reg_criterion(ef_pred, ef_true) + cls_criterion(cls_logit, cls_true)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running_loss += loss.item() * imgs.size(0); n_seen += imgs.size(0)

        if (time.time() - last_checkpoint_time) > CHECKPOINT_EVERY_MINUTES * 60:
            save_state(model, optimizer, epoch, step + 1, note='mid-epoch')
            last_checkpoint_time = time.time()

    resume_step = 0  # only skip steps on the first (resumed) epoch
    val_metrics = evaluate(val_loader)
    print(f"Epoch {epoch}: train_loss={running_loss/max(n_seen,1):.3f} val_mae={val_metrics['mae']:.2f} "
          f"val_auc={val_metrics['auc']:.3f} val_f1={val_metrics['f1']:.3f}")
    log_path = os.path.join(PROJECT_ROOT, 'logs', f'{EXP_NAME}.jsonl')
    with open(log_path, 'a') as f:
        f.write(json.dumps({'epoch': epoch, 'time': time.time(), 'val_mae': val_metrics['mae'],
                             'val_auc': val_metrics['auc'], 'val_f1': val_metrics['f1']}) + '\n')
    save_state(model, optimizer, epoch + 1, 0, note='epoch-end')

print('Training run complete (or reached N_EPOCHS). Re-run this cell after a disconnect to resume automatically.')

## MC-Dropout evaluation + risk-coverage curve on the full validation set

Same method as notebook 03, now with real statistical power (hundreds of val videos instead of 36).

In [ ]:
T_PASSES = 20

def mc_dropout_predict(model, loader, T=T_PASSES):
    model.eval()
    model.enable_mc_dropout()
    all_ef_true, all_cls_true, all_quality = [], [], []
    ef_pass_list = [[] for _ in range(T)]
    cls_pass_list = [[] for _ in range(T)]
    with torch.no_grad():
        for imgs, ef_true, cls_true, quality in loader:
            imgs = imgs.to(device)
            all_ef_true.extend(ef_true.numpy()); all_cls_true.extend(cls_true.numpy()); all_quality.extend(quality.numpy())
            for t in range(T):
                ef_pred, cls_logit = model(imgs)
                ef_pass_list[t].extend(ef_pred.cpu().numpy())
                cls_pass_list[t].extend(torch.sigmoid(cls_logit).cpu().numpy())
    ef_preds = np.array(ef_pass_list); cls_probs = np.array(cls_pass_list)
    return {
        'ef_true': np.array(all_ef_true), 'cls_true': np.array(all_cls_true), 'quality': np.array(all_quality),
        'ef_pred_mean': ef_preds.mean(axis=0), 'ef_pred_std': ef_preds.std(axis=0),
        'cls_prob_mean': cls_probs.mean(axis=0), 'cls_prob_std': cls_probs.std(axis=0),
    }

mc_results = mc_dropout_predict(model, val_loader)
corr = np.corrcoef(mc_results['quality'], mc_results['ef_pred_std'])[0, 1]
print(f'N validation videos: {len(mc_results["ef_true"])}')
print(f'Correlation quality_score vs EF uncertainty (expect negative): {corr:.3f}')

In [ ]:
import matplotlib.pyplot as plt

order = np.argsort(mc_results['ef_pred_std'])
n = len(order)
coverages = np.arange(1.0, 0.39, -0.05)
rows = []
for cov in coverages:
    k = max(10, int(n * cov))
    keep_idx = order[:k]
    mae_at_cov = mean_absolute_error(mc_results['ef_true'][keep_idx], mc_results['ef_pred_mean'][keep_idx])
    try:
        auc_at_cov = roc_auc_score(mc_results['cls_true'][keep_idx], mc_results['cls_prob_mean'][keep_idx])
    except ValueError:
        auc_at_cov = float('nan')
    rows.append({'coverage': cov, 'n_kept': k, 'mae': mae_at_cov, 'auc': auc_at_cov})

rc_df = pd.DataFrame(rows)
print(rc_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rc_df['coverage'], rc_df['mae'], marker='o'); axes[0].invert_xaxis()
axes[0].set_xlabel('Coverage'); axes[0].set_ylabel('EF MAE'); axes[0].set_title('Risk-coverage: EF (full val set)')
axes[1].plot(rc_df['coverage'], rc_df['auc'], marker='o', color='darkorange'); axes[1].invert_xaxis()
axes[1].set_xlabel('Coverage'); axes[1].set_ylabel('HFrEF AUC'); axes[1].set_title('Risk-coverage: HFrEF (full val set)')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'figures', 'risk_coverage_curve_full.png'), dpi=150)
plt.show()

## Notes

- If a session disconnects mid-training, just re-open this notebook and re-run all cells from the top -- frame extraction, quality scoring, and training all resume automatically from where they left off.
- Once val metrics look stable and the risk-coverage curve is monotonic (with real statistical power now), this is a genuine result worth writing up -- unlike the subsample run, which was purely a feasibility check.
- Remaining roadmap items once this looks good: run on the held-out **test** split only once (never used for tuning), compute bootstrap confidence intervals, and compare against the ProtoASNet numbers identified in the novelty audit.